In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder,RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,f1_score,classification_report
from sklearn.model_selection import train_test_split

In [2]:
data=pd.read_csv('loan_dataset.csv')

In [3]:
df=data.copy()
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,NaN,No,9600000,29900000,12,778,2400000.0,17600000.0,NaN,8000000.0,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,NaN,2200000.0,8800000.0,3300000.0,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000.0,NaN,33300000.0,12800000.0,Rejected
3,4,3,NaN,No,8200000,30700000,8,467,18200000.0,3300000.0,23300000.0,7900000.0,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000.0,8200000.0,29400000.0,5000000.0,Rejected


In [4]:
df.drop('loan_id',axis=1,inplace=True)

In [5]:
df.isnull().sum()

no_of_dependents              0
education                   640
self_employed               341
income_annum                  0
loan_amount                   0
loan_term                     0
cibil_score                   0
residential_assets_value    384
commercial_assets_value     128
luxury_assets_value         256
bank_asset_value            426
loan_status                   0
dtype: int64

In [6]:
numerical=[]
categorical=[]
for col in df:
    if df[col].dtype=='O':
        categorical.append(col)
        
    else:
        numerical.append(col)
        

In [7]:
print(categorical)
print()
print(numerical)

['education', 'self_employed', 'loan_status']

['no_of_dependents', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']


# <center>====Handling Missing Values====<center> 

# <center>====Random Value Imputation On Categorical columns====<center>

In [8]:
df[categorical].isnull().sum()

education        640
self_employed    341
loan_status        0
dtype: int64

In [9]:
cat_missing_col=df[categorical].iloc[:,:-1]
cat_missing_col.head()

,education,self_employed
0,NaN,No
1,Not Graduate,Yes
2,Graduate,No
3,NaN,No
4,Not Graduate,Yes


In [10]:
def randomvalueimpuation_cat(data,columns):
    df=data
    for col in columns:
        non_missing=df[col].dropna().values
        df[col]=df[col].apply(lambda x:np.random.choice(non_missing)if pd.isnull(x) else x)

In [11]:
randomvalueimpuation_cat(df,cat_missing_col)

In [12]:
df[categorical].isnull().sum()

education        0
self_employed    0
loan_status      0
dtype: int64

# <center>====Random Value Imputation On Numerical columns====<center>

In [13]:
df[numerical].isnull().sum()

no_of_dependents              0
income_annum                  0
loan_amount                   0
loan_term                     0
cibil_score                   0
residential_assets_value    384
commercial_assets_value     128
luxury_assets_value         256
bank_asset_value            426
dtype: int64

In [14]:
num_missing_col=df[numerical].iloc[:,5:]
num_missing_col.head()

,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
0,2400000.0,17600000.0,NaN,8000000.0
1,NaN,2200000.0,8800000.0,3300000.0
2,7100000.0,NaN,33300000.0,12800000.0
3,18200000.0,3300000.0,23300000.0,7900000.0
4,12400000.0,8200000.0,29400000.0,5000000.0


In [15]:
def randomvalueimputer_num(data,columns):
    df=data
    for col in columns:
        missing_vals=df[col].isnull().sum()
        pool=df[col].dropna().sample(missing_vals).values
        df[col][df[col].isnull()]=pool

In [16]:
randomvalueimputer_num(df,num_missing_col)

C:\Users\manish singh\AppData\Local\Temp\ipykernel_17236\2780033506.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col][df[col].isnull()]=pool
C:\Users\manish singh\AppData\Local\Temp\ipykernel_17236\2780033506.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col][df[col].isnull()]=pool
C:\Users\manish singh\AppData\Local\Temp\ipykernel_17236\2780033506.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col][df[col].i

In [17]:
df[numerical].isnull().sum()

no_of_dependents            0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
dtype: int64

# <center>====Handling Outliers====<center>

In [18]:
outlier_cols=['residential_assets_value','commercial_assets_value','bank_asset_value']

In [19]:
def iqr_method(data,columns):
    data=df
    for col in columns:
        p25=df[col].quantile(0.25)
        p75=df[col].quantile(0.75)



        iqr=p75-p25
        uf=p75+1.5*iqr
        lf=p25-1.5*iqr

        df[(df[col]>uf) | (df[col]<lf)].shape

        df[(df[col]<uf) & (df[col]>lf)].shape

        df[col]=np.where(df[col]>uf,uf,np.where(df[col]<lf,lf,df[col]))




In [20]:
iqr_method(df,outlier_cols)

In [21]:
df.head()

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,2,Graduate,No,9600000,29900000,12,778,2400000.0,17050000.0,14100000.0,8000000.0,Approved
1,0,Not Graduate,Yes,4100000,12200000,8,417,4000000.0,2200000.0,8800000.0,3300000.0,Rejected
2,3,Graduate,No,9100000,29700000,20,506,7100000.0,9200000.0,33300000.0,12800000.0,Rejected
3,3,Graduate,No,8200000,30700000,8,467,18200000.0,3300000.0,23300000.0,7900000.0,Rejected
4,5,Not Graduate,Yes,9800000,24200000,20,382,12400000.0,8200000.0,29400000.0,5000000.0,Rejected


# <center>====Incoding and Scaling====<center>

In [22]:
one=OneHotEncoder()
rs=RobustScaler()

In [23]:
col_trans=ColumnTransformer(transformers=[('rs',RobustScaler(),numerical),
                                          ('ohe',OneHotEncoder(drop='first',handle_unknown='ignore')
                                           ,categorical)],
                                remainder='passthrough')

In [24]:
col_trans

ColumnTransformer(remainder='passthrough',
                  transformers=[('rs', RobustScaler(),
                                 ['no_of_dependents', 'income_annum',
                                  'loan_amount', 'loan_term', 'cibil_score',
                                  'residential_assets_value',
                                  'commercial_assets_value',
                                  'luxury_assets_value', 'bank_asset_value']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore'),
                                 ['education', 'self_employed',
                                  'loan_status'])])

In [25]:
df_inc=col_trans.fit_transform(df)

In [26]:
df_incoded=pd.DataFrame(df_inc,columns=col_trans.get_feature_names_out())
df_incoded.head()

,rs__no_of_dependents,rs__income_annum,rs__loan_amount,rs__loan_term,rs__cibil_score,rs__residential_assets_value,rs__commercial_assets_value,rs__luxury_assets_value,rs__bank_asset_value,ohe__education_ Not Graduate,ohe__self_employed_ Yes,ohe__loan_status_ Rejected
0,-0.333333,0.937500,1.115942,0.2,0.603390,-0.340659,2.119048,-0.035211,0.760870,0.0,0.0,0.0
1,-1.000000,-0.208333,-0.166667,-0.2,-0.620339,-0.164835,-0.238095,-0.408451,-0.260870,1.0,1.0,1.0
2,0.000000,0.833333,1.101449,1.0,-0.318644,0.175824,0.873016,1.316901,1.804348,0.0,0.0,1.0
3,0.000000,0.645833,1.173913,-0.2,-0.450847,1.395604,-0.063492,0.612676,0.739130,0.0,0.0,1.0
4,0.666667,0.979167,0.702899,1.0,-0.738983,0.758242,0.714286,1.042254,0.108696,1.0,1.0,1.0


In [27]:
df_incoded.columns=df.columns

In [28]:
df_incoded.head()

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,-0.333333,0.937500,1.115942,0.2,0.603390,-0.340659,2.119048,-0.035211,0.760870,0.0,0.0,0.0
1,-1.000000,-0.208333,-0.166667,-0.2,-0.620339,-0.164835,-0.238095,-0.408451,-0.260870,1.0,1.0,1.0
2,0.000000,0.833333,1.101449,1.0,-0.318644,0.175824,0.873016,1.316901,1.804348,0.0,0.0,1.0
3,0.000000,0.645833,1.173913,-0.2,-0.450847,1.395604,-0.063492,0.612676,0.739130,0.0,0.0,1.0
4,0.666667,0.979167,0.702899,1.0,-0.738983,0.758242,0.714286,1.042254,0.108696,1.0,1.0,1.0


# <center>====Building Model====</center>

In [29]:
x=df_incoded.drop('loan_status',axis=1)
y=df_incoded['loan_status']

In [30]:
x_train,x_test,y_train,y_test=train_test_split(x,y,train_size=0.3)

In [31]:
x_train.shape,x_test.shape

((1280, 11), (2989, 11))

In [32]:
y_train.shape,y_test.shape

((1280,), (2989,))